In [3]:
import numpy as np
import pandas as pd
import scipy

from IPython.display import clear_output

import re
from pathlib import Path
import requests
import random
import time
import scipy
from bs4 import BeautifulSoup
import gc
import json
import glob

from tqdm import tqdm
import networkx as nx

In [4]:
url_df = pd.read_csv('../data/261028_clean_urls.csv')

How many journals with associated urls?

In [260]:
url_df['global_uid'].nunique()

21064

In [5]:
url_df[url_df['global_uid'] == 13767]

,service,global_uid,service_uid,url,clean_url,n_pred_services
27101,esji,13767,esji_7750.0,https://dergipark.org.tr/tr/pub/buefd,dergipark.org.tr,1


In [6]:
urls = url_df[~url_df['clean_url'].astype(str).str.contains(':')]['clean_url'].value_counts().index

In [8]:
with open('../tokens/builtwith_api_key.txt', 'r') as f:
    api_key = f.read()

In [6]:
collect_results = []
troublesome_urls = []
count = 0
for url in tqdm(urls):
    count += 1
    more_results = True
    next_skip = 0
    while more_results:
        try:
            pathy = Path('../data/260906_builtwith_api_calls/' + url + '_' + str(next_skip) + '.json')
            if pathy.exists():
                with open(pathy, 'r') as f:
                    response_text = f.read()
            else:
                builtwith_call = 'https://api.builtwith.com/rv4/api.json'
                params = {'KEY':api_key, 'LOOKUP':url, 'OFFSET':next_skip} # 'IP':'yes'
                time.sleep(1)
                response = requests.get(builtwith_call, params=params)
                response_text = response.text
                if response.status_code == 200:
                    pass
                else:
                    more_results = False
                    continue
        except:
            print('Trouble with url ' + url)
            troublesome_urls.append(url)
            
        with open('../data/260906_builtwith_api_calls/' + url + '_' + str(next_skip) + '.json', 'w') as f:
            json.dump(json.loads(response_text), f)
            
        errors = json.loads(response_text)['Errors']
        if len(errors) > 0: # invalid domain
            more_results = False
            troublesome_urls.append(url)
            continue
            
        more_results = json.loads(response_text)['more_results']
        next_skip = json.loads(response_text)['next_skip']
        if next_skip >= 5000:
            more_results = False

        identifier_objects = pd.json_normalize(json.loads(response_text)['Relationships'])['Identifiers'].values
        for id_obj in identifier_objects:
            df = pd.json_normalize(id_obj)
            df = df.rename(columns={'Value':'id_value', 
                       'Type':'id_type',
                       'First':'first_id_query_domain', 
                       'Last':'last_id_query_domain', 
                       'Matches':'matches'})
            df['query_domain'] = url
            df = df.reset_index().rename(columns={'index':'id_index'})

            if 'matches' in df.columns:
                collect_match_dfs = []
                for index, row in df.dropna(subset=['matches']).iterrows():
                    match_df = pd.json_normalize(row['matches'])
                    match_df.columns = ['match_domain', 'first_id_match_domain', 'last_id_match_domain', 'overlap']
                    match_df = match_df.rename(columns={'Domain':'match_domain', 
                           'First':'first_id_match_domain', 
                           'Last':'last_id_match_domain', 
                           'Overlap':'overlap'})
                    match_df['id_index'] = index
                    collect_match_dfs.append(match_df)
                match_df = pd.concat(collect_match_dfs)
                df = pd.merge(df, match_df, on='id_index', how='outer')
                collect_results.append(df)
            else:
                collect_results.append(df)

100%|██████████████████████████████████████████████████████████████████████████| 12468/12468 [4:48:26<00:00,  1.39s/it]


In [266]:
#pd.concat(collect_results).to_csv('../data/260906_120000_progress.csv', index=False)
builtwith_df = pd.read_csv('../../fake_impact_factors/data/260906_120000_progress.csv')

In [267]:
indirect_matches = pd.merge(url_df[['global_uid', 'clean_url']], 
         builtwith_df, 
         left_on='clean_url', 
         right_on='query_domain')[['global_uid', 'match_domain', 'id_type', 'id_value', 'overlap']].drop_duplicates()
indirect_matches = indirect_matches[indirect_matches['id_type'] != 'IP'].dropna()
indirect_matches.columns = ['global_uid', 'domain', 'how_domain_linked', 'id_value', 'overlap']

In [268]:
indirect_matches = indirect_matches[indirect_matches['id_value'] != '0']

In [269]:
direct_matches = url_df[['global_uid', 'clean_url', 'n_pred_services']].drop_duplicates()
direct_matches.columns = ['global_uid', 'domain', 'how_domain_linked']
direct_matches['how_domain_linked'] = 'same_domain'

In [270]:
edge_df = pd.merge(direct_matches[['global_uid', 'domain']], 
         pd.concat([direct_matches, indirect_matches]), on='domain')\
.rename(columns={'global_uid_x':'source', 'global_uid_y':'target'}).copy()

In [271]:
new_source = []
new_target = []
for source, target, index in zip(edge_df['source'], edge_df['target'], edge_df.index):
    if source > target:
        new_source.append(target)
        new_target.append(source)
    else:
        new_source.append(source)
        new_target.append(target)
edge_df['source'] = new_source
edge_df['target'] = new_target

In [272]:
edge_df['how_domain_linked'] = edge_df['how_domain_linked'].astype('category')\
.cat.set_categories(edge_df['how_domain_linked'].value_counts().index, ordered=True)

In [273]:
# prefer if journals share same domain

edge_df = edge_df[edge_df['source'] != edge_df['target']]\
.sort_values('how_domain_linked').drop_duplicates(['source', 'target'], keep='first')

In [274]:
edge_df['how_domain_linked'].value_counts()

same_domain    521210
CA-PUB          99593
UA              39574
AT               3699
GTM              1874
HRB               541
CRN               399
BAI               285
CF                223
FB                 74
YX                 56
GP                 48
SC                 48
EIN                28
NR                 18
ST                 14
TK                  8
CNPJ                5
RTB                 4
LC                  3
AL                  2
MGID                1
51LA                1
HJ                  0
OP                  0
OX                  0
CE                  0
MP                  0
Name: how_domain_linked, dtype: int64

In [275]:
edge_df['same_domain'] = edge_df['how_domain_linked'].isin(['same_domain'])

How many journals share same domain with another journal?

In [283]:
a,b = edge_df[edge_df['same_domain']][['source', 'target']].T.values
len(set(a) | set(b))

11118

In [284]:
a,b = edge_df[~edge_df['same_domain']][['source', 'target']].T.values
len(set(a) | set(b))

4049

In [178]:
edge_df.to_csv('../data/261028_builtwith_edgelist.csv', index=False)

In [21]:
G = nx.from_pandas_edgelist(edge_df)

In [24]:
# node attributes

n_pred_services = pd.read_csv('../data/261028_n_pred_services.csv')
n_pred_services_indexed = pd.read_csv('../data/261028_n_pred_services_indexed_journals.csv')

In [236]:
node_df = n_pred_services[['global_uid']]
node_df.columns = ['Id']
node_df['indexed'] = node_df['Id'].isin(n_pred_services_indexed['global_uid'])

In [237]:
one_row_per_journal_df = pd.read_csv('../data/261028_one_row_per_journal.csv')

In [238]:
one_row_per_journal_df['Id'] = one_row_per_journal_df['global_uid']
node_props_df = one_row_per_journal_df[one_row_per_journal_df['Id'].isin(node_df['Id'])]\
[['Id', 'titles', 'issns', 'urls', 'services', 'n_pred_services']]

In [239]:
node_df = pd.merge(node_props_df, node_df, on='Id')

In [240]:
node_df['indexed'].sum()

3752

How many OMICS group journals?

In [241]:
imedpub_urls = url_df[url_df['clean_url'].astype(str).str.contains('imedpub')]['clean_url'].unique()

In [286]:
# imedpub has many subdomains
omics = ['alliedacademies.org', 'andrewjohnpublishing.com', 'ashdin.com', 
 'conferenceseries.com', 'euroscicon.com', 'hilarispublisher.com', 
 'imedpub.com', 'iomcworld.org', 'longdom.org', 
 'omicsonline.org', 'omicsgroup.org', 'omicsgroup.com', 
 'scitechnol.com', 'walshmedicalmedia.com', 'pulsus.com', 
         'primescholars.com', 'rroij.com', 'tsijournals.com']
print(len(omics))
omics = omics + list(imedpub_urls)

18


In [285]:
[print(x) for x in omics]

alliedacademies.org
andrewjohnpublishing.com
ashdin.com
conferenceseries.com
euroscicon.com
hilarispublisher.com
imedpub.com
iomcworld.org
longdom.org
omicsonline.org
omicsgroup.org
omicsgroup.com
scitechnol.com
walshmedicalmedia.com
pulsus.com
primescholars.com
rroij.com
tsijournals.com
imedpub.com
psychopathology.imedpub.com
datamining.imedpub.com
biochem-molbio.imedpub.com
cheminformatics.imedpub.com
herbal-medicine.imedpub.com
diversityhealthcare.imedpub.com
asthma-and-bronchitis.imedpub.com
autoimmunediseases.imedpub.com
pancreas.imedpub.com
blood-pressure.imedpub.com
pediatrics.imedpub.com
clinical-psychiatry.imedpub.com
obstetrics.imedpub.com
bone.imedpub.com
colorectal-cancer.imedpub.com
clinical-pediatrics-dermatology.imedpub.com
internalmedicine.imedpub.com
clinical-nutrition.imedpub.com
polymerscience.imedpub.com
ejbio.imedpub.com
childhood-developmental-disorders.imedpub.com
reproductive-endocrinology-infertility.imedpub.com
anaesthesia-painmedicine.imedpub.com
hospital-med

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [243]:
omics_uids = url_df[url_df['clean_url'].isin(omics)]['global_uid'].unique()

In [244]:
len(omics_uids)

627

In [249]:
a, b= edge_df[(edge_df['source'].isin(omics_uids) | edge_df['target'].isin(omics_uids))][['source', 'target']].T.values
omics_adjacent = (set(a) | set(b)) - set(omics_uids)

In [250]:
len(omics_adjacent)

268

In [253]:
node_df['omics'] = node_df['Id'].isin(omics_uids)
node_df['omics_adjacent'] = node_df['Id'].isin(omics_adjacent)

In [254]:
node_df.to_csv('../data/261028_builtwith_node_properties.csv', index=False)

In [32]:
n_pred_services[n_pred_services['service'] >= 4]['global_uid']

17          17
18          18
22          22
25          25
29          29
         ...  
12997    12997
13137    13137
14161    14161
14468    14468
14534    14534
Name: global_uid, Length: 1464, dtype: int64

In [33]:
int_list = [864]
url_df[url_df['global_uid'].isin(int_list)]

,service,global_uid,service_uid,url,clean_url,n_pred_services
1950,sjif,864,sjif_18837,http://www.omicsonline.org/physical-chemistry-...,omicsonline.org,4
1951,esji,864,esji_6394.0,https://www.longdom.org/physical-chemistry-bio...,longdom.org,4
1952,cosmos,864,cosmos_3387,http://www.omicsonline.org,omicsonline.org,4
1953,drji,864,drji_2161-0398,https://www.longdom.org/physical-chemistry-bio...,longdom.org,4


In [95]:
# most indexed journal
one_row_per_journal_df.set_index('global_uid').loc[[4385, 15667]].values

array([['Asian Pacific Journal of Health Sciences (APJHS), Asian pacific Journal of Health Sciences, ASIAN PACIFIC JOURNAL OF HEALTH SCIENCES, Asian Pacific Journal of Health Sciences',
        'http://www.apjhs.com/, nan, https://apjhs.com/index.php/apjhs/about, http://www.apjhs.com',
        '2350-0964, 2349-0659',
        'cosmos, drji, ici, ifj, iifs, esji, ifsij, sjif', 8.0, 4385],
       ['International Journal of Recent Advances in science and technology (IJRAST), International Journal of Recent Advances in science and technology',
        'www.ijrast.com, http://www.ijrast.com', '2395-2318',
        'drji, cosmos', 2.0, 15667]], dtype=object)

In [94]:
# Leon Publications
# connected to https://plagiarismremovalservices.com/
one_row_per_journal_df.set_index('global_uid').loc[[4956, 3989, 5101, 636]].values

array([['The Pharmaceutical and Chemical Journal',
        'http://www.tpcj.org, http://tpcj.org/', '2349-7092',
        'drji, iifs, cosmos, esji, sjif', 5.0, 4956],
       ['Journal of Scientific and Engineering Research',
        'http://jsaer.com/, http://www.jsaer.com', '2394-2630',
        'cosmos, iifs, sjif, esji', 4.0, 3989],
       ['Chemistry Research Journal, CHEMISTRY RESEARCH JOURNAL',
        'http://www.chemrj.org, http://chemrj.org/, http://chemj.org/, http://chemrj.org',
        '2455-8990', 'cosmos, iifs, sjif, esji', 4.0, 5101],
       ['European Journal of Advances In Engineering and Technology, European Journal of Advances in Engineering and Technology',
        'http://www.ejaet.com/, http://www.ejaet.com', '2394-658X',
        'iifs, sjif, esji', 3.0, 636]], dtype=object)

In [96]:
one_row_per_journal_df.set_index('global_uid').loc[[4908]].values

array([['IMPACT: International Journal of Research in Engineering & Technology, IMPACT : International Journal of Research in Engineering & Technology',
        'nan, http://www.impactjournals.us/journals.php?jtype=2&id=77, http://www.impactjournals.us/, http://www.impactjournals.us/journals.php?jtype=2&',
        '2321-8843', 'ifj, drji, sjif, iifs', 4.0, 4908]], dtype=object)

In [97]:
one_row_per_journal_df.set_index('global_uid').loc[[6167]].values

array([['Furniture and Wooden Material Research Journal, Furniture and Wooden Materials Research Journal, Mobilya ve Ahşap Malzeme Araştırmaları Dergisi',
        'nan, http://dergipark.gov.tr/mamad, http://www.dergipark.gov.tr',
        '2636-8625', 'drji, ici, scopus_indexed, esji, cosmos, sjif',
        5.0, 6167]], dtype=object)

In [98]:
one_row_per_journal_df.set_index('global_uid').loc[[1714, 7158, 2177]].values

array([['Pharmaceutical Methods, Pharmaceutical methods',
        'http://www.phmethods.net, http://www.phmethods.net/',
        '2229-4708, 2229-4716', 'sjif, cosmos', 2.0, 1714],
       ['Women’s Health Medicine',
        'http://www.womenshealthjournal.co.uk/index.php/womenshealth',
        '1744-1870', 'sjif', 1.0, 7158],
       ['INDIAN JOURNAL OF RHEUMATOLOGY, Indian Journal of Rheumatology',
        'nan, http://www.indianjrheumatol.com/', '0973-3701, 0973-3698',
        'scopus_indexed, doaj_indexed, wos_indexed, sjif', 1.0, 2177]],
      dtype=object)

In [100]:
one_row_per_journal_df.set_index('global_uid').loc[[9616, 9884]].values

array([['Educational Process: International Journal, EDUCATIONAL PROCESS: INTERNATIONAL JOURNAL, Educational process : international journal',
        'nan, http://edupij.com/ojs/index.php/edu/index, http://www.edupij.com',
        '2147-0901, 2564-8020',
        'scopus_deindexed, doaj_indexed, esji, cosmos, jufo_indexed',
        2.0, 9616],
       ['ГњNД°VERSД°TEPARK BГјlten, Üniversitepark Bülten',
        'nan, http://www.unibulletin.com/', '2147-351X',
        'doaj_deindexed, esji', 1.0, 9884]], dtype=object)

In [106]:
one_row_per_journal_df.set_index('global_uid').loc[[15140, 16253, 6085, 8895, 8185]].values

array([['International Journal of Education and Psychological Research, INTERNATIONAL JOURNAL OF EDUCATION AND PSYCHOLOGICAL RESEARCH',
        'http://www.ijepr.org', '2279-0179', 'iifs, cosmos', 2.0, 15140],
       ['INTERNATIONAL JOURNAL OF COMPUTATIONAL ENGINEERING AND MANAGEMENT IJCEM',
        'http://www.ijcem.org', '2230-7893', 'cosmos', 1.0, 16253],
       ['World Journal of  Current Medical And Pharmaceutical Research, World Journal of Current Medical and Pharmaceutical Research',
        'http://www.wjcmpr.com/index.php, https://wjcmpr.com/index.php, https://www.wjcmpr.com/index.php',
        '2582-0222', 'drji, iifs, sjif, esji', 4.0, 6085],
       ['International journal of leadership and public sector reform',
        'http://journals.osu.edu.et/index.php/ijlpsr',
        '3005-8368, 3006-4945', 'sjif', 1.0, 8895],
       ['international journal of engineering science and advanced technology, International Journal of Engineering Science and Advanced Technology, Internatio

In [200]:
one_row_per_journal_df.set_index('global_uid').loc[[15759]].values

array([['JOURNAL OF CLINICAL OBSTETRICS, GYNECOLOGY & INFERTILITY, Journal of Clinical Obstetrics, Gynecology & Infertility',
        'http://www.remedypublications.com/clinical-obstetrics-gynecology-infertility/, http://www.remedypublications.com',
        '2573-8755', 'drji, cosmos', 2.0, 15759]], dtype=object)

In [159]:
node_df

,Id,titles,issns,services,n_pred_services,indexed
0,4385,Asian Pacific Journal of Health Sciences (APJH...,"2350-0964, 2349-0659","cosmos, drji, ici, ifj, iifs, esji, ifsij, sjif",8.0,False
1,4283,"Journal of Pharma Research, Journal of Pharma ...",2319-5622,"drji, ifj, iifs, cosmos, esji, ifsij, doaj_dei...",7.0,False
2,79,International Journal of Innovations in Engine...,2394-3696,"drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,False
3,8967,"internation amuk tamuk, International Journal ...","2349-5138, 2348-1269","drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,False
4,1254,EMPEROR INTERNATIONAL JOURNAL OF FINANCE AND M...,2395-5929,"drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,False
...,...,...,...,...,...,...
21249,10415,PEP Educational Assessment,"2580-782X, 2580-7641",esji,1.0,False
21250,10412,Dissertare: Revista de Investigación en Cienci...,2542-3177,"doaj_indexed, esji",1.0,True
21251,10386,Journal of Medives: Journal of Mathematics Edu...,"2549-5070, 2549-8231","doaj_indexed, esji",1.0,True
21252,10401,JIPFRI: Jurnal Inovasi Pendidikan Fisika dan R...,"2549-9076, 2549-905X",esji,1.0,False


In [201]:
one_row_per_journal_df

,global_uid,titles,urls,issns,services,n_pred_services,Id
0,4385,Asian Pacific Journal of Health Sciences (APJH...,"http://www.apjhs.com/, nan, https://apjhs.com/...","2350-0964, 2349-0659","cosmos, drji, ici, ifj, iifs, esji, ifsij, sjif",8.0,4385
1,4283,"Journal of Pharma Research, Journal of Pharma ...","nan, http://jprinfo.com/index.php, http://www....",2319-5622,"drji, ifj, iifs, cosmos, esji, ifsij, doaj_dei...",7.0,4283
2,79,International Journal of Innovations in Engine...,"http://www.ijiert.org/, nan, http://www.ijiert...",2394-3696,"drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,79
3,8967,"internation amuk tamuk, International Journal ...","nan, https://www.ijrar.org/, http://www.ijrar....","2349-5138, 2348-1269","drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,8967
4,1254,EMPEROR INTERNATIONAL JOURNAL OF FINANCE AND M...,"http://www.eijfmr.com, nan, http//www.eijfmr.com",2395-5929,"drji, ifj, iifs, cosmos, esji, ifsij, sjif",7.0,1254
...,...,...,...,...,...,...,...
78481,40334,"ACS Symposium Series, Acs symposium series",NaN,"1947-5918, 0097-6156","scopus_indexed, jufo_indexed",0.0,40334
78482,40335,ACS/IEEE International Conference on Computer ...,NaN,"2161-5330, 2161-5322","scopus_indexed, jufo_indexed",0.0,40335
78483,40336,"Acsms health and fitness journal, ACSM's Healt...",NaN,"1091-5397, 1536-593X","wos_indexed, scopus_indexed, jufo_indexed",0.0,40336
78484,40337,Acta academiae regiae gustavi adolphi lxxxvii,NaN,0065-0897,jufo_indexed,0.0,40337


# Are any of the indices connected to journals?

In [291]:
urls = ['cosmosimpactfactor.com', 
        'olddrji.lbp.world', 
        'esjindex.org', 
        'internationalcitationindex.org', 
        'impactfactorjournals.com', 
        'ifsij.com', 
        'impactfactorservice.com', 
        'sjifactor.com']

In [289]:
collect_results = []
troublesome_urls = []
count = 0
for url in tqdm(urls):
    count += 1
    more_results = True
    next_skip = 0
    while more_results:
        try:
            pathy = Path('../data/261111_builtwith_api_calls/' + url + '_' + str(next_skip) + '.json')
            if pathy.exists():
                with open(pathy, 'r') as f:
                    response_text = f.read()
            else:
                builtwith_call = 'https://api.builtwith.com/rv4/api.json'
                params = {'KEY':api_key, 'LOOKUP':url, 'OFFSET':next_skip} # 'IP':'yes'
                time.sleep(1)
                response = requests.get(builtwith_call, params=params)
                response_text = response.text
                if response.status_code == 200:
                    pass
                else:
                    more_results = False
                    continue
        except:
            print('Trouble with url ' + url)
            troublesome_urls.append(url)
            
        with open('../data/261111_builtwith_api_calls/' + url + '_' + str(next_skip) + '.json', 'w') as f:
            json.dump(json.loads(response_text), f)
            
        errors = json.loads(response_text)['Errors']
        if len(errors) > 0: # invalid domain
            more_results = False
            troublesome_urls.append(url)
            continue
            
        more_results = json.loads(response_text)['more_results']
        next_skip = json.loads(response_text)['next_skip']
        if next_skip >= 5000:
            more_results = False

        identifier_objects = pd.json_normalize(json.loads(response_text)['Relationships'])['Identifiers'].values
        for id_obj in identifier_objects:
            df = pd.json_normalize(id_obj)
            df = df.rename(columns={'Value':'id_value', 
                       'Type':'id_type',
                       'First':'first_id_query_domain', 
                       'Last':'last_id_query_domain', 
                       'Matches':'matches'})
            df['query_domain'] = url
            df = df.reset_index().rename(columns={'index':'id_index'})

            if 'matches' in df.columns:
                collect_match_dfs = []
                for index, row in df.dropna(subset=['matches']).iterrows():
                    match_df = pd.json_normalize(row['matches'])
                    match_df.columns = ['match_domain', 'first_id_match_domain', 'last_id_match_domain', 'overlap']
                    match_df = match_df.rename(columns={'Domain':'match_domain', 
                           'First':'first_id_match_domain', 
                           'Last':'last_id_match_domain', 
                           'Overlap':'overlap'})
                    match_df['id_index'] = index
                    collect_match_dfs.append(match_df)
                match_df = pd.concat(collect_match_dfs)
                df = pd.merge(df, match_df, on='id_index', how='outer')
                collect_results.append(df)
            else:
                collect_results.append(df)

100%|████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:10<00:00,  1.31s/it]


In [292]:
index_matches = pd.concat(collect_results)

In [300]:
matched_uids = url_df[url_df['clean_url'].isin(index_matches.dropna(subset=['match_domain'])['match_domain'])]['global_uid'].unique()

In [305]:
one_row_per_journal_df[one_row_per_journal_df['global_uid'].isin(matched_uids)]

,global_uid,titles,urls,issns,services,n_pred_services,Id
916,2553,Indian Streams Research Journal,"nan, http://oldisrj.lbp.world/, http://www.isr...",2230-7850,"drji, ifj, iifs, doaj_deindexed, sjif",4.0,2553
1374,333,"GOLDEN RESEARCH THOUGHTS, Golden Research Thou...","nan, http://aygrt.isrj.net/, http://oldgrt.lbp...",2231-5063,"drji, ifj, iifs, doaj_deindexed, sjif",4.0,333
1491,4556,"E-LIBRARY SCIENCE RESEARCH JOURNAL, E Library ...","http://www.lsrj.in/, http://oldlsrj.lbp.world,...",2319-8435,"drji, sjif, iifs",3.0,4556
1938,3825,"Review of Research, Review of research, Review...","nan, http://www.ror.isrj.net/, http://www.ror....",2249-894X,"drji, doaj_deindexed, sjif, iifs",3.0,3825
3693,2583,Physical Education and Sports Research Journal...,"http://oldpesrj.lbp.world/, http://pe.lsrj.in/...",2277-3665,"drji, sjif, iifs",3.0,2583
7246,15139,"Reviews of Progress, Reviews Of Progress, REVI...","http://oldroprogress.lbp.world/Home.aspx, nan,...",2321-3485,"drji, doaj_deindexed, iifs",2.0,15139
7680,15193,"Reviews of Literature, Review of Literature","http://oldrol.lbp.world/default.aspx, http://r...",2347-2723,"drji, iifs",2.0,15193
14606,18048,Tactful Management,"nan, http://oldtm.lbp.world/",2319-7943,"drji, doaj_deindexed",1.0,18048
